# N-BEATS - Deep Learning Time Series Model
imports, set up mlflow.

In [6]:
import numpy as np
import pandas as pd
import mlflow

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS

from preprocessing import load_raw, build_features, to_long_format, wmae

np.random.seed(42)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("NBEATS_Training")

<Experiment: artifact_location=('file:C:/Users/aluda '
 'bekurishvili/Desktop/home/projects/store_sales_forecasting/mlruns/2'), creation_time=1783781188104, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783781188104, lifecycle_stage='active', name='NBEATS_Training', tags={}, trace_location=None, workspace='default'>

## 1. Load & Reshape Data for neuralforecast

In [7]:
train, test, features, stores = load_raw()
train_full = build_features(train, features, stores)
long_df = to_long_format(train_full)

print(long_df.shape)
long_df.head()

(421570, 4)


,unique_id,ds,y,IsHoliday
0,10_1,2010-02-05,40212.84,False
1,10_1,2010-02-12,67699.32,True
2,10_1,2010-02-19,49748.33,False
3,10_1,2010-02-26,33601.22,False
4,10_1,2010-03-05,36572.44,False


## 2. Filter Series

In [8]:
H = 39
INPUT_SIZE = 52
MIN_LEN = INPUT_SIZE + H

long_df_all = long_df.copy()  
series_len = long_df.groupby("unique_id").size()
keep_ids = series_len[series_len >= MIN_LEN].index

print(f"series total: {series_len.shape[0]}")
print(f"series kept (len >= {MIN_LEN}): {len(keep_ids)}")
print(f"series dropped (too short): {series_len.shape[0] - len(keep_ids)}")

long_df = long_df[long_df["unique_id"].isin(keep_ids)].reset_index(drop=True)
print("rows after filtering:", long_df.shape[0])

with mlflow.start_run(run_name="NBEATS_Preprocessing"):
    mlflow.log_param("horizon_h", H)
    mlflow.log_param("input_size", INPUT_SIZE)
    mlflow.log_param("min_series_length", MIN_LEN)
    mlflow.log_metric("series_total", series_len.shape[0])
    mlflow.log_metric("series_kept", len(keep_ids))
    mlflow.log_metric("series_dropped", series_len.shape[0] - len(keep_ids))
    mlflow.log_metric("rows_kept", long_df.shape[0])

series total: 3331
series kept (len >= 91): 2900
series dropped (too short): 431
rows after filtering: 410069


## 3. Time-Based Cross-Validation

In [9]:
MAX_STEPS = 200 

model = NBEATS(
    h=H,
    input_size=INPUT_SIZE,
    max_steps=MAX_STEPS,
    scaler_type="standard",
    random_seed=42,
    enable_progress_bar=False,
    logger=False,
)
nf = NeuralForecast(models=[model], freq="W-FRI")

cv_df = nf.cross_validation(df=long_df, n_windows=None, test_size=H)
cv_df = cv_df.merge(
    long_df[["unique_id", "ds", "IsHoliday"]], on=["unique_id", "ds"], how="left"
)
cv_df.head()

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
7.2 K     Non-trainable params
2.6 M     Total params
10.321    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode
c:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_steps=200` reached.
GPU available: False, used: False
TPU available:

,unique_id,ds,cutoff,NBEATS,y,IsHoliday
0,10_1,2012-02-03,2012-01-27,31688.773438,36444.00,False
1,10_1,2012-02-10,2012-01-27,40372.035156,50434.11,True
2,10_1,2012-02-17,2012-01-27,44372.382812,74930.33,False
3,10_1,2012-02-24,2012-01-27,35631.445312,28751.57,False
4,10_1,2012-03-02,2012-01-27,35199.804688,30525.88,False


In [10]:
val_wmae = wmae(cv_df["y"], cv_df["NBEATS"], cv_df["IsHoliday"])
val_mae = (cv_df["y"] - cv_df["NBEATS"]).abs().mean()

print(f"Validation WMAE: {val_wmae:.2f}")
print(f"Validation MAE:  {val_mae:.2f}")

with mlflow.start_run(run_name="NBEATS_CV"):
    mlflow.log_param("horizon_h", H)
    mlflow.log_param("input_size", INPUT_SIZE)
    mlflow.log_param("max_steps", MAX_STEPS)
    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("val_mae", val_mae)

Validation WMAE: 1825.50
Validation MAE:  1777.94


## 4. Final Model Training (Full History) & Pipeline Export

In [11]:
final_keep_ids = series_len[series_len >= INPUT_SIZE].index
long_df_final = long_df_all[long_df_all["unique_id"].isin(final_keep_ids)].reset_index(drop=True)
print(f"series kept for final fit (len >= {INPUT_SIZE}): {len(final_keep_ids)} (vs {len(keep_ids)} for CV)")

final_model = NBEATS(
    h=H,
    input_size=INPUT_SIZE,
    max_steps=MAX_STEPS,
    scaler_type="standard",
    random_seed=42,
    enable_progress_bar=False,
    logger=False,
)
nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
nf_final.fit(df=long_df_final)

SAVE_PATH = "models/nbeats_nf"

nf_final.save(path=SAVE_PATH, overwrite=True, save_dataset=True)
print("saved to", SAVE_PATH)

Seed set to 42


series kept for final fit (len >= 52): 2991 (vs 2900 for CV)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
7.2 K     Non-trainable params
2.6 M     Total params
10.321    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode
c:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_steps=200` reached.


saved to models/nbeats_nf


In [12]:
class NBEATSPipeline(mlflow.pyfunc.PythonModel):
    """Wraps a fitted NeuralForecast(NBEATS) model to run on raw
    Store/Dept/Date input (e.g. Kaggle test.csv), unprocessed. NBEATS has no
    exogenous inputs, so prediction only needs unique_id/ds to join back onto —
    the model itself forecasts the next `h` steps from each series' history."""

    def load_context(self, context):
        self.nf = NeuralForecast.load(path=context.artifacts["nf_model"])

    def predict(self, context, model_input: pd.DataFrame):
        from preprocessing import add_unique_id

        df = add_unique_id(model_input.copy())
        df["ds"] = pd.to_datetime(df["Date"])

        preds = self.nf.predict()
        preds = preds.rename(columns={"NBEATS": "Weekly_Sales"})

        out = df[["Store", "Dept", "Date", "unique_id", "ds"]].merge(
            preds, on=["unique_id", "ds"], how="left"
        )
        return out[["Store", "Dept", "Date", "Weekly_Sales"]]


with mlflow.start_run(run_name="NBEATS_Final") as run:
    mlflow.log_param("horizon_h", H)
    mlflow.log_param("input_size", INPUT_SIZE)
    mlflow.log_param("max_steps", MAX_STEPS)
    mlflow.log_metric("val_wmae", val_wmae)  # carried over from the CV run above
    mlflow.log_metric("final_series_kept", len(final_keep_ids))

    mlflow.pyfunc.log_model(
        artifact_path="nbeats_pipeline",
        python_model=NBEATSPipeline(),
        artifacts={"nf_model": SAVE_PATH},
        code_paths=["preprocessing.py"],
    )
    print("Logged pipeline under run:", run.info.run_id)

2026/07/12 05:22:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 05:22:52 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting
2026/07/12 05:22:52 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/12 05:22:52 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


2026/07/12 05:22:53 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting
2026/07/12 05:22:53 INFO mlflow.utils.environment: Detected uv project at c:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting. Attempting to export requirements via 'uv export'.
2026/07/12 05:22:53 INFO mlflow.utils.uv_utils: Exported 252 dependencies via uv
2026/07/12 05:22:53 INFO mlflow.utils.environment: Successfully exported 252 requirements from uv project. Skipping package capture based inference.
2026/07/12 05:22:55 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/07/12 05:22:55 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting
2026/07/12 05:22:55 INFO mlflow.uti

Logged pipeline under run: eb019ac9206443c6b4217b831f873597
